# SO4/T4 spatial parameter review

The earlier starter is the current comparator for joint across-row, along-row and depth
allocation. Select `local111` in the input cell to review the candidate from the earlier
geometry checks. The exact inputs and ensemble results come from the saved local study.

Run this notebook with the project's Python 3.10 environment. To read the saved study
files from another checkout, set `DEEPROOTGEN_RESEARCH_ROOT` to the local project folder
containing the research bundles and `.runtime`. A clean clone uses the labelled synthetic
example. Save an executed copy under `.runtime` when working with field results.

In [ ]:
from hashlib import sha256
from pathlib import Path
import json
import os
import sys

from IPython.display import Image, display
import pandas as pd

cwd = Path.cwd()
checkout = next(
    (path for path in (cwd, *cwd.parents)
     if (path / "deeprootgen/data_model/canonical.py").is_file()),
    None,
)
if checkout is None:
    raise RuntimeError("Start Jupyter in the DeepRootGen checkout.")
sys.path.insert(0, str(checkout))

from deeprootgen.data_model.resolution import load_json_config, resolve_config
from deeprootgen.run_records import tracked_run

research_root = Path(os.environ.get("DEEPROOTGEN_RESEARCH_ROOT", str(checkout))).expanduser()
spatial = research_root / "Calvin_SO4_Parameter_Exploration_2026-11-18/Spatial_Evaluation_2026-11-19"

study_id = "so4-t4"
cases = {
    "starter": "Calvin_Structural_Optimization_2026-11-18/SO4_T4_direction_start.schema4.private.json",
    "local111": "Calvin_SO4_Parameter_Exploration_2026-11-18/SO4_T4_start.schema4.private.json",
}
case = "starter"
synthetic_example = "examples/research/native_steps_synthetic.json"

config_path = research_root / cases[case]
field_case = config_path.is_file()
if not field_case:
    config_path = checkout / synthetic_example
    print("Synthetic example. Set DEEPROOTGEN_RESEARCH_ROOT to load the saved study case.")

config_bytes = config_path.read_bytes()
config = load_json_config(config_bytes.decode("utf-8"))
resolved = resolve_config(config)
print(f"{case if field_case else 'synthetic'}  |  schema {config['schema_version']}  |  seed {resolved.seed}")
print("Configuration", config_path)
print("Input file SHA-256", sha256(config_bytes).hexdigest())

## Review the input

The first table lists supplied scalar settings and their resolver metadata. Feature
tables show the order-specific laws. Empirical length arrays are summarized by their
size and bounds, and daughter counts by their conditional range and mean. Open the
referenced JSON for exact values and arrays.

In [ ]:
consumers = resolved.description["consumers"]
parameter_rows = []
for name, value in config["parameters"].items():
    details = consumers.get(name, {})
    parameter_rows.append({
        "parameter": name,
        "value": value,
        "units": details.get("units", ""),
        "operation": details.get("operation", ""),
    })
display(pd.DataFrame(parameter_rows))

order_rows, feature_rows = [], []
for name, feature in config.get("features", {}).items():
    if not feature["enabled"]:
        continue
    settings = feature.get("settings", {})
    other_settings = {k: v for k, v in settings.items() if k != "by_order"}
    feature_rows.append({"feature": name, "settings": other_settings})
    for order, rule in settings.get("by_order", {}).items():
        detail = dict(rule) if isinstance(rule, dict) else {"length_cm": rule}
        if detail.get("law") == "discrete_cdf":
            lengths = detail.pop("lengths_cm")
            detail.pop("cumulative_probabilities")
            detail.update(support_points=len(lengths), min_cm=min(lengths), max_cm=max(lengths))
        if "daughter_probabilities" in detail:
            probabilities = detail.pop("daughter_probabilities")
            supported = [i for i, value in enumerate(probabilities, 1) if value > 0]
            detail.update(daughter_count_min=min(supported, default=None), daughter_count_max=max(supported, default=None),
                          conditional_mean_daughters=sum(i * value for i, value in enumerate(probabilities, 1)))
        order_rows.append({"order": int(order), "feature": name, "settings": detail})

if feature_rows:
    display(pd.DataFrame(feature_rows))
if order_rows:
    display(pd.DataFrame(order_rows).sort_values(["order", "feature"]))

## Joint location and depth in the saved SO4/T4 comparison

Total variation (TV) compares root-length shares within the aligned soil volume.
Lower TV means closer allocation. Read the joint score and the horizontal-by-depth
projections together. The comparison uses simulations and the original T4 observations.

In [ ]:
scores_path = spatial / "so4_spatial_scores.csv"
if field_case and scores_path.is_file():
    scores = pd.read_csv(scores_path)
    columns = ["candidate", "seeds", "voxel_cm", "joint_xyz_tv",
               "across_depth_tv", "along_depth_tv", "depth_tv"]
    display(scores.loc[:, columns].sort_values(["voxel_cm", "joint_xyz_tv"]))
    figure_path = spatial / "so4_spatial_depth_maps.png"
    if figure_path.is_file():
        display(Image(filename=str(figure_path)))
else:
    print("Saved SO4/T4 spatial scores are available with the local study bundle.")

## Replay one seed

Set `run_preview` to `True` to generate one realization with the saved input. Set
`preview_seed` to replay another seed. The output directory contains the input,
geometry, per-order summaries, a 3D view, and the tracked run record. This is a
one-seed preview. The spatial scores above use the original ensemble.

In [ ]:
from uuid import uuid4

run_preview = False
preview_seed = None  # Supply an integer to sample another realization.
if run_preview:
    run_root = checkout / ".runtime/notebook-review-runs"
    label = f"{study_id}-{case if field_case else 'synthetic'}"
    run_dir = run_root / f"{label}-{uuid4().hex[:8]}"

    def set_seed(data):
        data["random_seed"] = preview_seed

    record = tracked_run(
        config_path, run_dir, label=label, ledger=run_root / "runs.jsonl",
        edit=set_seed if preview_seed is not None else None,
    )
    summary = json.loads((run_dir / "root_summary.json").read_bytes())
    display(pd.DataFrame(summary["by_order"]))
    display(pd.DataFrame(summary["diameter_classes"]).T)
    print("Saved files", run_dir)
    print("Run identity", record["model_identity"])